<a href="https://colab.research.google.com/github/Lianaulya/Practical-Statistics-for-Data-Scientists/blob/main/Chapter3_StatisticalExperiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| | |
|---|---|
| **Nama** | Muammar Yaldi Fadhil |
| **NIM** | 101032300019 |
| **Kelas** | TK-47-03 |
| **Mata Kuliah** | Machine Learning |

---

# Chapter 3 – Statistical Experiments and Significance Testing
**Book:** *Practical Statistics for Data Scientists* (O'Reilly)

Topics:
- A/B Testing
- Hypothesis Tests (p-values, null hypothesis)
- Resampling Permutation Tests
- t-Tests
- Multiple Testing
- ANOVA and F-statistics
- Chi-Square Test
- Power & Sample Size


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE_DATA = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data"
print("Libraries loaded!")

## 1. A/B Testing

An **A/B test** (controlled experiment) compares two versions of something (A and B)
to determine which performs better.

**Procedure:**
1. Define a clear metric (conversion rate, click-through rate, etc.)
2. Randomly assign subjects to group A (control) and B (treatment)
3. Run the experiment and collect data
4. Test whether the observed difference is statistically significant

**Null hypothesis $H_0$**: There is no difference between A and B.
**Alternative hypothesis $H_a$**: There is a difference.


In [ ]:
# A/B Test — web page session times
session_times = pd.read_csv(f"{BASE_DATA}/web_page_data.csv")
session_times.columns = ["Page", "Time"]
print(session_times.groupby("Page")["Time"].describe())

In [ ]:
# Visualise group distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, page, color in zip(axes, ["Page A", "Page B"], ["steelblue", "coral"]):
    data = session_times[session_times["Page"] == page]["Time"]
    data.hist(bins=20, ax=ax, color=color, edgecolor="black")
    ax.axvline(data.mean(), color="black", linestyle="--",
               label=f"Mean = {data.mean():.1f}s")
    ax.set_title(f"{page} – Session Time")
    ax.set_xlabel("Time (seconds)")
    ax.legend()

plt.tight_layout()
plt.show()

## 2. Hypothesis Tests and P-values

The **p-value** is the probability of observing a result at least as extreme as
the observed statistic, **assuming the null hypothesis is true**.

- If $p < \alpha$ (usually 0.05): reject $H_0$ → "statistically significant"
- If $p \geq \alpha$: fail to reject $H_0$

> **Caution:** A small p-value does not imply practical significance. Always check effect size!

In [ ]:
# Permutation test for A/B session times
page_a = session_times[session_times["Page"] == "Page A"]["Time"].values
page_b = session_times[session_times["Page"] == "Page B"]["Time"].values

observed_diff = page_b.mean() - page_a.mean()
print(f"Observed difference (B - A): {observed_diff:.4f} seconds")

np.random.seed(42)
combined = np.concatenate([page_a, page_b])
n_a = len(page_a)
permutation_diffs = []
for _ in range(2000):
    perm = np.random.permutation(combined)
    permutation_diffs.append(perm[n_a:].mean() - perm[:n_a].mean())

p_value = np.mean(np.abs(permutation_diffs) >= np.abs(observed_diff))
print(f"Permutation p-value: {p_value:.4f}")

In [ ]:
# Visualize permutation distribution
fig, ax = plt.subplots(figsize=(9, 4))
pd.Series(permutation_diffs).hist(bins=40, ax=ax, color="steelblue", edgecolor="black")
ax.axvline(observed_diff, color="red", linewidth=2, label=f"Observed diff = {observed_diff:.3f}")
ax.axvline(-observed_diff, color="orange", linewidth=2, linestyle="--")
ax.set_title("Permutation Distribution – H₀: No difference between pages")
ax.set_xlabel("Difference in Means (B − A)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. t-Test

The **two-sample t-test** tests whether the means of two groups differ:

$$t = \frac{\bar{X}_A - \bar{X}_B}{\sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}}$$

With Welch correction for unequal variances.

In [ ]:
# Two-sample t-test
t_stat, p_val = stats.ttest_ind(page_b, page_a, equal_var=False)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_val:.4f}")
print(f"Conclusion:  {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'}")

## 4. ANOVA and F-Statistic

Analysis of Variance (**ANOVA**) tests whether the means of **3+ groups** differ.

$$F = \frac{\text{Between-group variance (MSB)}}{\text{Within-group variance (MSW)}}$$

Large $F$ → evidence against $H_0$ (that all group means are equal).

In [ ]:
# ANOVA: web page session times across four sessions
four_sessions = pd.read_csv(f"{BASE_DATA}/four_sessions.csv")
print(four_sessions.head())
print("\nGroup means:")
print(four_sessions.groupby("Page")["Time"].mean())

In [ ]:
# One-way ANOVA
groups = [group["Time"].values for _, group in four_sessions.groupby("Page")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value:     {p_val:.4f}")
print(f"Conclusion:  {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'}")

# Box plot of groups
fig, ax = plt.subplots(figsize=(8, 4))
four_sessions.boxplot("Time", by="Page", ax=ax)
ax.set_title("Session Time by Page Version")
ax.set_xlabel("Page")
ax.set_ylabel("Time (seconds)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Chi-Square Test

The **chi-square test** checks whether categorical variables are independent:

$$\chi^2 = \sum_i \frac{(O_i - E_i)^2}{E_i}$$

where $O_i$ = observed, $E_i$ = expected count under independence.

In [ ]:
# Chi-square test: click-through rates by headline
click_rate = pd.read_csv(f"{BASE_DATA}/click_rates.csv")
print(click_rate)

In [ ]:
# Contingency table and chi-square test
contingency = click_rate.pivot_table(index="Headline", columns="Click", values="Rate")
contingency = click_rate.groupby("Headline")["Rate"].sum().reset_index()

# Simple chi-square on the click-rate table
pivot = click_rate.pivot_table(index="Headline", columns="Click", values="Rate", fill_value=0)
chi2, p, dof, expected = stats.chi2_contingency(pivot.values)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value:              {p:.4f}")
print(f"Degrees of freedom:   {dof}")

## 6. Power and Sample Size

**Statistical power** (1 − β) is the probability of correctly rejecting $H_0$ when it is false.

Key relationships:
- Higher power → need larger sample size
- Smaller effect size → need larger sample size
- Smaller $\alpha$ → need larger sample size

Minimum sample size per group (two-sample t-test):

$$n \approx \frac{2\sigma^2 (z_{\alpha/2} + z_{\beta})^2}{\delta^2}$$

In [ ]:
# Power analysis: what sample size for 80% power?
from scipy.stats import norm

alpha = 0.05
power = 0.80
effect_size = 0.3  # standard deviations

z_alpha2 = norm.ppf(1 - alpha/2)   # 1.96
z_beta   = norm.ppf(power)          # 0.84

n_required = int(np.ceil(2 * ((z_alpha2 + z_beta) / effect_size) ** 2))
print(f"Required n per group: {n_required}")
print(f"(Effect size = {effect_size} SD, alpha = {alpha}, power = {power})")

# Plot power vs. sample size
ns = np.arange(10, 500, 5)
powers = []
for n in ns:
    z = effect_size * np.sqrt(n/2) - z_alpha2
    powers.append(norm.cdf(z))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ns, powers, color="steelblue", linewidth=2)
ax.axhline(0.80, color="red", linestyle="--", label="80% power")
ax.axvline(n_required, color="orange", linestyle="--", label=f"n = {n_required}")
ax.set_title("Statistical Power vs. Sample Size")
ax.set_xlabel("Sample Size (per group)")
ax.set_ylabel("Power")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Concept | Key Formula |
|---------|------------|
| p-value | P(data as extreme \| H₀ true) |
| t-test | $t = (\bar{X}_A - \bar{X}_B) / SE_{diff}$ |
| ANOVA F | MSB / MSW |
| Chi-square | $\sum (O-E)^2/E$ |
| Power | $P(\text{reject }H_0 \mid H_a \text{ true})$ |
